# Week 14: LLM Fine-Tuning (PEFT), Quantization & Acceleration
## Notebook 06: FlashAttention Mechanics & Online Softmax from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Standard Transformer Self-Attention is bounded by GPU Memory Bandwidth (Memory-Bound, not Compute-Bound):
- Computing $\text{Softmax}(Q K^T / \sqrt{d}) V$ requires writing the full $N \times N$ attention matrix to High Bandwidth Memory (HBM/VRAM).
- For sequence length $N = 32,768$, an FP16 attention matrix requires **2 Gigabytes of VRAM per head per layer**, causing GPU Out-Of-Memory (OOM) crashes.

In 2022, Tri Dao (Stanford) introduced **FlashAttention**:
- **IO-Awareness:** Never materializes the full $N \times N$ attention matrix in GPU HBM!
- Splits $Q, K, V$ into small blocks that fit entirely in ultra-fast **on-chip SRAM** (19 TB/s vs 2 TB/s HBM).
- Computes attention incrementally using **Online Softmax**!

---

### 2. Mathematical Foundations: Online Softmax
Standard softmax requires computing the maximum across the *entire* row before exponentiating:
$$m = \max_j x_j, \quad d = \sum_j e^{x_j - m}, \quad p_i = \frac{e^{x_i - m}}{d}$$

#### Online Softmax Formulation (Milakov & Gimelshein):
Given two blocks of inputs $x^{(1)}$ and $x^{(2)}$:
1. Local max: $m_1 = \max(x^{(1)}), \quad m_2 = \max(x^{(2)})$.
2. New global max: $m_{\text{new}} = \max(m_1, m_2)$.
3. Update running denominator incrementally:
   $$d_{\text{new}} = d_1 e^{m_1 - m_{\text{new}}} + d_2 e^{m_2 - m_{\text{new}}}$$
This allows calculating exact softmax in a single streaming pass across blocks without storing intermediate results!

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Standard 2-Pass Softmax
x = np.array([2.0, 5.0, 1.0, 8.0, 3.0, 7.0])

# Pass 1: Global Max
m_global = np.max(x)
# Pass 2: Exp and Sum
d_global = np.sum(np.exp(x - m_global))
standard_softmax = np.exp(x - m_global) / d_global

print("Input Vector:", x)
print("Standard Softmax Output:\n", np.round(standard_softmax, 4))

# Step 2: Implement FlashAttention Online Tiled Softmax across 2 Blocks
# Block 1: [2.0, 5.0, 1.0]
# Block 2: [8.0, 3.0, 7.0]
block1 = x[:3]
block2 = x[3:]

# Process Block 1
m1 = np.max(block1)
d1 = np.sum(np.exp(block1 - m1))
# Scaled partial unnormalized output for Block 1
out1 = np.exp(block1 - m1)

# Process Block 2 (Streaming update without loading Block 1 back from memory!)
m2 = np.max(block2)
d2 = np.sum(np.exp(block2 - m2))
out2 = np.exp(block2 - m2)

# Combine using Online Softmax Update Rules
m_new = max(m1, m2)
d_new = d1 * np.exp(m1 - m_new) + d2 * np.exp(m2 - m_new)

# Rescale previous outputs
out1_rescaled = out1 * np.exp(m1 - m_new) / d_new
out2_rescaled = out2 * np.exp(m2 - m_new) / d_new

online_softmax = np.concatenate([out1_rescaled, out2_rescaled])

print("\nOnline Streaming Softmax Output:\n", np.round(online_softmax, 4))
print("Exact Match:", np.allclose(standard_softmax, online_softmax))

### 3. Step-by-Step Code Walkthrough
1. **Block Partitioning:** Input is partitioned into tiles matching GPU SRAM capacity.
2. **Dynamic Rescaling:** When Block 2 reveals a higher maximum ($m_2=8.0 > m_1=5.0$), Block 1's contribution is dynamically discounted by $e^{m_1 - m_2}$.
3. **Exact Accuracy:** Online softmax produces identical outputs to standard softmax while reducing memory footprint from $O(N^2) \to O(N)$.

---

### 4. Industry Applications & Unsloth
- **Unsloth:** Re-engineers PyTorch backward autograd kernels with manual Triton/CUDA implementations of FlashAttention-2, achieving **2x to 5x faster fine-tuning** with 80% less VRAM on consumer GPUs.

---

### 5. Top Interview Questions & Answers
**Q1: Why is FlashAttention called "IO-Aware"?**
> **Answer:** Standard algorithms account only for computation FLOPs. FlashAttention considers the GPU memory hierarchy, recognizing that reading/writing from HBM (DRAM) is 10x slower than SRAM compute. By fusing matrix multiplication and softmax inside SRAM tiles, it minimizes HBM memory transfers.

**Q2: How does FlashAttention compute gradients during backpropagation without storing the $N \times N$ attention matrix?**
> **Answer:** Instead of saving the massive attention matrix to HBM during the forward pass, FlashAttention recomputes attention blocks on-the-fly during the backward pass in SRAM from stored $Q, K, V$ chunks, trading minor recompute FLOPs for massive memory bandwidth savings.